<a href="https://colab.research.google.com/github/AnanyaAsthana/Hadoop-CUDA-Lab/blob/main/cudaLab5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile matrix_threads_seconds.cu
#include <stdio.h>
#include <cuda.h>
#include <time.h>

__global__ void matrixMulGPU(int *A, int *B, int *C, int N){
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if(row < N && col < N){
        int sum = 0;
        for(int k = 0; k < N; k++){
            sum += A[row*N + k] * B[k*N + col];
        }
        C[row*N + col] = sum;
    }
}

void matrixMulCPU(int *A, int *B, int *C, int N){
    for(int i=0;i<N;i++){
        for(int j=0;j<N;j++){
            int sum = 0;
            for(int k=0;k<N;k++){
                sum += A[i*N+k] * B[k*N+j];
            }
            C[i*N+j] = sum;
        }
    }
}

int main(){

    int N;
    printf("Enter matrix size: ");
    scanf("%d",&N);

    int size = N*N*sizeof(int);

    int *A,*B,*C_cpu,*C_gpu;

    A=(int*)malloc(size);
    B=(int*)malloc(size);
    C_cpu=(int*)malloc(size);
    C_gpu=(int*)malloc(size);

    for(int i=0;i<N*N;i++){
        A[i]=rand()%10;
        B[i]=rand()%10;
    }

    // CPU timing
    clock_t start,end;
    start = clock();

    matrixMulCPU(A,B,C_cpu,N);

    end = clock();
    double cpu_time = ((double)(end-start))/CLOCKS_PER_SEC;
    printf("CPU Time = %f seconds\n",cpu_time);

    int *dA,*dB,*dC;

    cudaMalloc(&dA,size);
    cudaMalloc(&dB,size);
    cudaMalloc(&dC,size);

    cudaMemcpy(dA,A,size,cudaMemcpyHostToDevice);
    cudaMemcpy(dB,B,size,cudaMemcpyHostToDevice);

    dim3 threads(16,16);
    dim3 blocks((N+15)/16,(N+15)/16);

    cudaEvent_t startg,stopg;
    cudaEventCreate(&startg);
    cudaEventCreate(&stopg);

    cudaEventRecord(startg);

    matrixMulGPU<<<blocks,threads>>>(dA,dB,dC,N);

    cudaEventRecord(stopg);
    cudaEventSynchronize(stopg);

    float gpu_time_ms;
    cudaEventElapsedTime(&gpu_time_ms,startg,stopg);

    double gpu_time_sec = gpu_time_ms / 1000.0;

    printf("GPU Time = %f seconds\n",gpu_time_sec);

    cudaMemcpy(C_gpu,dC,size,cudaMemcpyDeviceToHost);

    cudaFree(dA);
    cudaFree(dB);
    cudaFree(dC);

    free(A);
    free(B);
    free(C_cpu);
    free(C_gpu);

    return 0;
}

Writing matrix_threads_seconds.cu


In [ ]:
!nvcc matrix_threads_seconds.cu -o matrix_threads_seconds

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./matrix_threads_seconds

Enter matrix size: 100
CPU Time = 0.002793 seconds
GPU Time = 0.000195 seconds


In [ ]:
!./matrix_threads_seconds


Enter matrix size: 50
CPU Time = 0.000366 seconds
GPU Time = 0.000160 seconds


In [ ]:
!./matrix_threads_seconds

Enter matrix size: 40
CPU Time = 0.000195 seconds
GPU Time = 0.000221 seconds


In [ ]:
!./matrix_threads_seconds


Enter matrix size: 45
CPU Time = 0.000241 seconds
GPU Time = 0.000165 seconds


In [ ]:
%%writefile matrix_shared_compare.cu
#include <stdio.h>
#include <cuda.h>
#include <time.h>

#define TILE 16

// CPU Matrix Multiplication
void matrixMulCPU(int *A,int *B,int *C,int N){

    for(int i=0;i<N;i++){
        for(int j=0;j<N;j++){
            int sum = 0;

            for(int k=0;k<N;k++){
                sum += A[i*N+k] * B[k*N+j];
            }

            C[i*N+j] = sum;
        }
    }
}


// GPU Kernel using Shared Memory
__global__ void matrixMulShared(int *A,int *B,int *C,int N){

    __shared__ int As[TILE][TILE];
    __shared__ int Bs[TILE][TILE];

    int row = blockIdx.y*TILE + threadIdx.y;
    int col = blockIdx.x*TILE + threadIdx.x;

    int sum = 0;

    for(int t=0;t<(N+TILE-1)/TILE;t++){

        if(row<N && t*TILE+threadIdx.x<N)
            As[threadIdx.y][threadIdx.x] = A[row*N + t*TILE + threadIdx.x];
        else
            As[threadIdx.y][threadIdx.x] = 0;

        if(col<N && t*TILE+threadIdx.y<N)
            Bs[threadIdx.y][threadIdx.x] = B[(t*TILE+threadIdx.y)*N + col];
        else
            Bs[threadIdx.y][threadIdx.x] = 0;

        __syncthreads();

        for(int k=0;k<TILE;k++)
            sum += As[threadIdx.y][k] * Bs[k][threadIdx.x];

        __syncthreads();
    }

    if(row<N && col<N)
        C[row*N+col] = sum;
}


int main(){

    int N;
    printf("Enter matrix size: ");
    scanf("%d",&N);

    int size = N*N*sizeof(int);

    int *A,*B,*C_cpu,*C_gpu;

    A = (int*)malloc(size);
    B = (int*)malloc(size);
    C_cpu = (int*)malloc(size);
    C_gpu = (int*)malloc(size);

    for(int i=0;i<N*N;i++){
        A[i] = rand()%10;
        B[i] = rand()%10;
    }

    // -------- CPU TIME --------
    clock_t start,end;

    start = clock();

    matrixMulCPU(A,B,C_cpu,N);

    end = clock();

    double cpu_time = ((double)(end-start))/CLOCKS_PER_SEC;

    printf("CPU Time = %f seconds\n",cpu_time);


    // -------- GPU PART --------
    int *dA,*dB,*dC;

    cudaMalloc(&dA,size);
    cudaMalloc(&dB,size);
    cudaMalloc(&dC,size);

    cudaMemcpy(dA,A,size,cudaMemcpyHostToDevice);
    cudaMemcpy(dB,B,size,cudaMemcpyHostToDevice);

    dim3 threads(TILE,TILE);
    dim3 blocks((N+TILE-1)/TILE,(N+TILE-1)/TILE);

    cudaEvent_t startg,stopg;

    cudaEventCreate(&startg);
    cudaEventCreate(&stopg);

    cudaEventRecord(startg);

    matrixMulShared<<<blocks,threads>>>(dA,dB,dC,N);

    cudaEventRecord(stopg);
    cudaEventSynchronize(stopg);

    float gpu_time_ms;

    cudaEventElapsedTime(&gpu_time_ms,startg,stopg);

    double gpu_time_sec = gpu_time_ms/1000.0;

    printf("GPU Shared Memory Time = %f seconds\n",gpu_time_sec);

    cudaMemcpy(C_gpu,dC,size,cudaMemcpyDeviceToHost);

    cudaFree(dA);
    cudaFree(dB);
    cudaFree(dC);

    free(A);
    free(B);
    free(C_cpu);
    free(C_gpu);

    return 0;
}

Writing matrix_shared_compare.cu


In [ ]:
!nvcc matrix_shared_compare.cu -o matrix_shared_compare

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./matrix_shared_compare

Enter matrix size: 50
CPU Time = 0.000332 seconds
GPU Shared Memory Time = 0.000182 seconds
